In [1]:
import os
# Enforce single GPU execution to prevent multi-GPU synchronization conflicts
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import math
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from safetensors.torch import load_file
from huggingface_hub import login, hf_hub_download

# ==============================================================================
# GLOBAL DEBUG CONTROL
# Set DEBUG = True for rapid pipeline verification (seconds), False for production.
# ==============================================================================
DEBUG = False

DEBUG_CONFIG = {
    "train_samples": 64,
    "dev_samples": 16,
    "test_samples": 8,
    "teacher_epochs": 1,
    "student_epochs": 1,
    "batch_size": 2,
    "log_interval": 2,
    "teacher_lr": 3e-4,
    "student_lr": 5e-4,
    "lorma_rank": 16,
    "lorma_alpha": 32,
    "kd_temperature": 2.0,
    "kd_alpha": 0.5,  # Weight for Hard Cross-Entropy Loss vs Soft KD Loss
    "max_doc_len": 768,
    "max_summ_len": 256,
    "gen_max_new_tokens": 128
}

FULL_CONFIG = {
    "train_samples": None,  # Full dataset
    "dev_samples": None,
    "test_samples": None,
    "teacher_epochs": 3,
    "student_epochs": 3,
    "batch_size": 2,
    "log_interval": 20,
    "teacher_lr": 3e-4,
    "student_lr": 5e-4,
    "lorma_rank": 16,
    "lorma_alpha": 32,
    "kd_temperature": 2.0,
    "kd_alpha": 0.5,
    "max_doc_len": 1536,
    "max_summ_len": 384,
    "gen_max_new_tokens": 256
}

CFG = DEBUG_CONFIG if DEBUG else FULL_CONFIG

print(f"=== Phase 6 Execution Mode: {'DEBUG (Fast Verification Run)' if DEBUG else 'PRODUCTION RUN'} ===")
print(f"Configuration: {CFG}\n")

=== Phase 6 Execution Mode: PRODUCTION RUN ===
Configuration: {'train_samples': None, 'dev_samples': None, 'test_samples': None, 'teacher_epochs': 3, 'student_epochs': 3, 'batch_size': 2, 'log_interval': 20, 'teacher_lr': 0.0003, 'student_lr': 0.0005, 'lorma_rank': 16, 'lorma_alpha': 32, 'kd_temperature': 2.0, 'kd_alpha': 0.5, 'max_doc_len': 1536, 'max_summ_len': 384, 'gen_max_new_tokens': 256}



In [ ]:
HF_TOKEN = os.environ["HF_TOKEN"]
login(token=HF_TOKEN)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MODEL_REPO_TEACHER = "A-240242/legal-modern-decoder-checkpoints" 
HF_TOKENIZER_REPO = "A-240242/legal-modern-decoder-checkpoints"

def load_summ_file(filename):
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO, 
        filename=f"Phase_6_Summ/{filename}", 
        repo_type="dataset", 
        token=HF_TOKEN
    )
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Downloading Phase 6 Abstractive Summarization Data...")
full_train_data = load_summ_file("Train/train.json")
test_data = load_summ_file("Test/test.json")

print(f"Loaded {len(full_train_data)} total training examples and {len(test_data)} test examples.")

# Construct a held-out validation split (10%) from train.json
np.random.seed(42)
indices = np.arange(len(full_train_data))
np.random.shuffle(indices)

val_size = max(1, int(0.10 * len(full_train_data)))
val_indices = set(indices[:val_size])

train_data = [full_train_data[i] for i in range(len(full_train_data)) if i not in val_indices]
dev_data = [full_train_data[i] for i in val_indices]

print(f"Split data into {len(train_data)} training samples and {len(dev_data)} validation samples.")

if DEBUG and CFG["train_samples"]:
    train_data = train_data[:CFG["train_samples"]]
    dev_data = dev_data[:CFG["dev_samples"]]
    test_data = test_data[:CFG["test_samples"]]
    print(f"\n[DEBUG] Subsampled train to {len(train_data)}, dev to {len(dev_data)}, and test to {len(test_data)} records.")

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

Loaded 7030 total training examples and 100 test examples.
Split data into 6327 training samples and 703 validation samples.


In [3]:
tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER_REPO, token=HF_TOKEN)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

class LegalSummarizationDataset(Dataset):
    """
    Causal Language Modeling Dataset for Abstractive Summarization[cite: 9].
    Prompt Format:
      "Document:\n{document}\n\nSummary:\n{summary}"
    Labels for document tokens are masked with -100 so loss is computed strictly on summary tokens.
    """
    def __init__(self, raw_records, tokenizer, max_doc_len=1536, max_summ_len=384, is_test=False):
        self.samples = []
        self.is_test = is_test
        
        for record in raw_records:
            doc_id = record.get("id")
            doc_text = record.get("document", "")
            
            if is_test:
                prompt = f"Document:\n{doc_text}\n\nSummary:\n"
                prompt_ids = tokenizer.encode(prompt, truncation=True, max_length=max_doc_len)
                self.samples.append({
                    "id": doc_id,
                    "input_ids": prompt_ids,
                    "prompt_text": prompt
                })
            else:
                summ_text = record.get("summary", "")
                
                prompt_prefix = f"Document:\n{doc_text}\n\nSummary:\n"
                prefix_ids = tokenizer.encode(prompt_prefix, truncation=True, max_length=max_doc_len)
                summ_ids = tokenizer.encode(summ_text, truncation=True, max_length=max_summ_len)
                
                if tokenizer.eos_token_id not in summ_ids:
                    summ_ids.append(tokenizer.eos_token_id)
                    
                full_input_ids = prefix_ids + summ_ids
                labels = [-100] * len(prefix_ids) + summ_ids
                
                self.samples.append({
                    "id": doc_id,
                    "input_ids": full_input_ids,
                    "labels": labels
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def summ_collate_fn(batch):
    if "labels" in batch[0]:
        input_ids_list = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
        labels_list = [torch.tensor(item["labels"], dtype=torch.long) for item in batch]
        doc_ids = [item["id"] for item in batch]
        
        padded_input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids_list, batch_first=True, padding_value=tokenizer.pad_token_id
        )
        padded_labels = torch.nn.utils.rnn.pad_sequence(
            labels_list, batch_first=True, padding_value=-100
        )
        attention_mask = (padded_input_ids != tokenizer.pad_token_id).long()
        
        return {
            "id": doc_ids,
            "input_ids": padded_input_ids,
            "attention_mask": attention_mask,
            "labels": padded_labels
        }
    else:
        input_ids_list = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
        doc_ids = [item["id"] for item in batch]
        
        padded_input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids_list, batch_first=True, padding_value=tokenizer.pad_token_id
        )
        attention_mask = (padded_input_ids != tokenizer.pad_token_id).long()
        
        return {
            "id": doc_ids,
            "input_ids": padded_input_ids,
            "attention_mask": attention_mask
        }

print("Building Phase 6 DataLoaders...")
train_dataset = LegalSummarizationDataset(train_data, tokenizer, max_doc_len=CFG["max_doc_len"], max_summ_len=CFG["max_summ_len"])
dev_dataset = LegalSummarizationDataset(dev_data, tokenizer, max_doc_len=CFG["max_doc_len"], max_summ_len=CFG["max_summ_len"])

train_loader = DataLoader(train_dataset, batch_size=CFG["batch_size"], shuffle=True, collate_fn=summ_collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=CFG["batch_size"], shuffle=False, collate_fn=summ_collate_fn)

print(f"Created {len(train_dataset)} train samples and {len(dev_dataset)} dev samples.")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

[transformers] You are using a model of type `modern_decoder` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

Building Phase 6 DataLoaders...
Created 6327 train samples and 703 dev samples.


In [4]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_position_embeddings: int = 2048, base: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, x: torch.Tensor, seq_len: int):
        t = torch.arange(seq_len, device=x.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos(), emb.sin()


def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    cos = cos[:q.shape[2], :].unsqueeze(0).unsqueeze(0)
    sin = sin[:q.shape[2], :].unsqueeze(0).unsqueeze(0)
    
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


class ModernDecoderAttention(nn.Module):
    """Modern Multi-Head Self-Attention with RoPE."""
    def __init__(self, hidden_size=768, num_heads=12):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, cos: torch.Tensor = None, sin: torch.Tensor = None):
        B, S, _ = x.shape
        
        q = self.q_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        
        if cos is not None and sin is not None:
            q, k = apply_rotary_pos_emb(q, k, cos, sin)
            
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        causal_mask = torch.full((S, S), float("-inf"), device=x.device)
        causal_mask = torch.triu(causal_mask, diagonal=1)
        scores = scores + causal_mask[None, None, :, :]
        
        if attention_mask is not None:
            padding_mask = (1.0 - attention_mask[:, None, None, :].float()) * -10000.0
            scores = scores + padding_mask
            
        probs = F.softmax(scores, dim=-1)
        attn_out = torch.matmul(probs, v)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, self.hidden_size)
        
        return self.o_proj(attn_out)


class SwiGLUMLP(nn.Module):
    """SwiGLU Feed-Forward Network."""
    def __init__(self, hidden_size=768, intermediate_size=2048):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class ModernDecoderLayer(nn.Module):
    def __init__(self, hidden_size=768, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.input_layernorm = RMSNorm(hidden_size)
        self.self_attn = ModernDecoderAttention(hidden_size, num_heads)
        self.post_attention_layernorm = RMSNorm(hidden_size)
        self.mlp = SwiGLUMLP(hidden_size, intermediate_size)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, cos: torch.Tensor = None, sin: torch.Tensor = None):
        residual = x
        x = self.input_layernorm(x)
        x = self.self_attn(x, attention_mask=attention_mask, cos=cos, sin=sin)
        x = residual + x
        
        residual = x
        x = self.post_attention_layernorm(x)
        x = self.mlp(x)
        x = residual + x
        return x


class ModernDecoderBackbone(nn.Module):
    def __init__(self, vocab_size, hidden_size=768, num_layers=6, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        self.layers = nn.ModuleList([
            ModernDecoderLayer(hidden_size, num_heads, intermediate_size)
            for _ in range(num_layers)
        ])
        self.norm = RMSNorm(hidden_size)
        self.rotary_emb = RotaryEmbedding(hidden_size // num_heads)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None):
        B, S = input_ids.shape
        x = self.embed_tokens(input_ids)
        cos, sin = self.rotary_emb(x, S)
        
        for layer in self.layers:
            x = layer(x, attention_mask=attention_mask, cos=cos, sin=sin)
            
        x = self.norm(x)
        return x


class CustomModernDecoderCausalLM(nn.Module):
    def __init__(self, vocab_size, hidden_size=768, num_layers=6, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.model = ModernDecoderBackbone(vocab_size, hidden_size, num_layers, num_heads, intermediate_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None, labels: torch.Tensor = None):
        hidden_states = self.model(input_ids, attention_mask=attention_mask)
        logits = self.lm_head(hidden_states)
        
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fn(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            
        return {"loss": loss, "logits": logits}

print("Custom Modern Decoder Causal LM Architecture written successfully.")

Custom Modern Decoder Causal LM Architecture written successfully.


In [5]:
class LormaAdapter(nn.Module):
    """Low-Rank Matrix Adaptation (LoRMA) Layer[cite: 10, 11]."""
    def __init__(self, in_features, out_features, rank=16, alpha=32):
        super().__init__()
        self.scaling = alpha / rank
        self.A = nn.Linear(in_features, rank, bias=False)
        self.B = nn.Linear(rank, out_features, bias=False)
        nn.init.normal_(self.A.weight, std=1e-3)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.scaling * self.B(self.A(x))


class LoRMAWrappedLinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, rank=16, alpha=32):
        super().__init__()
        self.base = base_linear
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False
            
        self.adapter = LormaAdapter(
            base_linear.in_features, base_linear.out_features, rank=rank, alpha=alpha
        )

    def forward(self, x):
        return self.base(x) + self.adapter(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================================================================
# 1. TEACHER MODEL INSTANTIATION (Full 6-Layer Model + Phase 1 Weights + LoRMA)
# ==============================================================================
teacher_model = CustomModernDecoderCausalLM(
    vocab_size=len(tokenizer), 
    hidden_size=768, 
    num_layers=6, 
    num_heads=12, 
    intermediate_size=2048
)

print("Fetching Phase 1 Modern Decoder Weights for Teacher...")
teacher_weight_path = hf_hub_download(repo_id=HF_MODEL_REPO_TEACHER, filename="model.safetensors", token=HF_TOKEN)
state_dict = load_file(teacher_weight_path)
teacher_model.load_state_dict(state_dict, strict=False)

# Freeze Teacher base weights and inject LoRMA adapters
for param in teacher_model.parameters():
    param.requires_grad = False

for param in teacher_model.lm_head.parameters():
    param.requires_grad = True

for layer in teacher_model.model.layers:
    for target_attr in ["q_proj", "k_proj", "v_proj", "o_proj"]:
        base_linear = getattr(layer.self_attn, target_attr)
        setattr(layer.self_attn, target_attr, LoRMAWrappedLinear(base_linear, rank=CFG["lorma_rank"], alpha=CFG["lorma_alpha"]))
        
    for target_attr in ["gate_proj", "up_proj", "down_proj"]:
        base_linear = getattr(layer.mlp, target_attr)
        setattr(layer.mlp, target_attr, LoRMAWrappedLinear(base_linear, rank=CFG["lorma_rank"], alpha=CFG["lorma_alpha"]))

for name, param in teacher_model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

teacher_model.to(device)

# ==============================================================================
# 2. STUDENT MODEL INSTANTIATION (Compact 3-Layer Model with ~50% parameters)
# ==============================================================================
student_model = CustomModernDecoderCausalLM(
    vocab_size=len(tokenizer), 
    hidden_size=768, 
    num_layers=3,  # 3 layers instead of 6 for lightweight student
    num_heads=12, 
    intermediate_size=2048
)

# Initialize student embeddings and LM head from teacher weights for fast convergence
student_model.model.embed_tokens.weight.data.copy_(teacher_model.model.embed_tokens.weight.data)
student_model.lm_head.weight.data.copy_(teacher_model.lm_head.weight.data)

student_model.to(device)

teacher_trainable_params = sum(p.numel() for p in teacher_model.parameters() if p.requires_grad)
student_total_params = sum(p.numel() for p in student_model.parameters())

print(f"\nArchitecture Readout:")
print(f"  - Teacher Model Total Parameters: {sum(p.numel() for p in teacher_model.parameters()):,}")
print(f"  - Teacher Trainable Parameters (LoRMA): {teacher_trainable_params:,}")
print(f"  - Student Model Total Parameters (Compact 3-Layer): {student_total_params:,}")

Fetching Phase 1 Modern Decoder Weights for Teacher...


model.safetensors: reconstructing file:   0%|          |  0.00B /  536MB            

model.safetensors: downloading bytes:           |  0.00B            


Architecture Readout:
  - Teacher Model Total Parameters: 93,030,144
  - Teacher Trainable Parameters (LoRMA): 25,976,832
  - Student Model Total Parameters (Compact 3-Layer): 70,391,040


In [6]:
CHECKPOINT_DIR = "./checkpoints_phase6_teacher_lorma"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
TEACHER_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "best_teacher_modern_lorma.pt")

teacher_epochs = CFG["teacher_epochs"]
teacher_optimizer = torch.optim.AdamW([p for p in teacher_model.parameters() if p.requires_grad], lr=CFG["teacher_lr"], weight_decay=0.01)

best_teacher_val_loss = float("inf")

print(f"=== TASK 1: Training Teacher Model with LoRMA ({teacher_epochs} Epochs) ===")

for epoch in range(teacher_epochs):
    teacher_model.train()
    total_train_loss = 0.0
    
    for step, batch in enumerate(train_loader):
        teacher_optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = teacher_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs["loss"]
        
        loss.backward()
        teacher_optimizer.step()
        
        total_train_loss += loss.item()
        
        if (step + 1) % CFG["log_interval"] == 0 or (step + 1) == len(train_loader):
            print(f"Teacher | Epoch {epoch+1}/{teacher_epochs} | Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
            
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation
    teacher_model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for batch in dev_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = teacher_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_val_loss += outputs["loss"].item()

    avg_val_loss = total_val_loss / len(dev_loader)
    print(f"Teacher | Epoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_teacher_val_loss or epoch == 0:
        best_teacher_val_loss = avg_val_loss
        trainable_state_dict = {
            k: v.cpu().clone() for k, v in teacher_model.state_dict().items() if v.requires_grad
        }
        torch.save({"epoch": epoch + 1, "trainable_state_dict": trainable_state_dict}, TEACHER_CHECKPOINT_PATH)
        print(f"  [+] Saved Best Teacher Checkpoint (Val Loss: {best_teacher_val_loss:.4f})")
        
    print("-" * 75)

=== TASK 1: Training Teacher Model with LoRMA (3 Epochs) ===
Teacher | Epoch 1/3 | Step 20/3164 | Loss: 7.0822
Teacher | Epoch 1/3 | Step 40/3164 | Loss: 7.4354
Teacher | Epoch 1/3 | Step 60/3164 | Loss: 6.7800
Teacher | Epoch 1/3 | Step 80/3164 | Loss: 6.4217
Teacher | Epoch 1/3 | Step 100/3164 | Loss: 6.2247
Teacher | Epoch 1/3 | Step 120/3164 | Loss: 5.3693
Teacher | Epoch 1/3 | Step 140/3164 | Loss: 5.6896
Teacher | Epoch 1/3 | Step 160/3164 | Loss: 5.2453
Teacher | Epoch 1/3 | Step 180/3164 | Loss: 5.2121
Teacher | Epoch 1/3 | Step 200/3164 | Loss: 5.6820
Teacher | Epoch 1/3 | Step 220/3164 | Loss: 5.6341
Teacher | Epoch 1/3 | Step 240/3164 | Loss: 5.4015
Teacher | Epoch 1/3 | Step 260/3164 | Loss: 5.2861
Teacher | Epoch 1/3 | Step 280/3164 | Loss: 5.5755
Teacher | Epoch 1/3 | Step 300/3164 | Loss: 5.5184
Teacher | Epoch 1/3 | Step 320/3164 | Loss: 5.5211
Teacher | Epoch 1/3 | Step 340/3164 | Loss: 5.2583
Teacher | Epoch 1/3 | Step 360/3164 | Loss: 5.3697
Teacher | Epoch 1/3 | Ste

In [7]:
# Load Best Teacher Weights for Distillation
if os.path.exists(TEACHER_CHECKPOINT_PATH):
    checkpoint = torch.load(TEACHER_CHECKPOINT_PATH, map_location=device)
    teacher_model.load_state_dict(checkpoint["trainable_state_dict"], strict=False)
    print(f"Loaded Best Teacher Checkpoint for Knowledge Distillation.")

teacher_model.eval()

STUDENT_CHECKPOINT_DIR = "./checkpoints_phase6_student_kd"
os.makedirs(STUDENT_CHECKPOINT_DIR, exist_ok=True)
STUDENT_CHECKPOINT_PATH = os.path.join(STUDENT_CHECKPOINT_DIR, "best_student_kd.pt")

student_epochs = CFG["student_epochs"]
student_optimizer = torch.optim.AdamW(student_model.parameters(), lr=CFG["student_lr"], weight_decay=0.01)

T = CFG["kd_temperature"]
alpha = CFG["kd_alpha"]
best_student_val_loss = float("inf")

print(f"\n=== TASK 2: Knowledge Distillation to Student Model ({student_epochs} Epochs) ===")

for epoch in range(student_epochs):
    student_model.train()
    total_train_loss = 0.0
    
    for step, batch in enumerate(train_loader):
        student_optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # 1. Forward pass through frozen Teacher
        with torch.no_grad():
            teacher_outputs = teacher_model(input_ids=input_ids, attention_mask=attention_mask)
            teacher_logits = teacher_outputs["logits"]
            
        # 2. Forward pass through Student
        student_outputs = student_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        student_logits = student_outputs["logits"]
        student_ce_loss = student_outputs["loss"]
        
        # 3. Compute Logit-Level KL-Divergence Distillation Loss over non-masked tokens
        shift_student = student_logits[..., :-1, :].contiguous()
        shift_teacher = teacher_logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        
        valid_mask = (shift_labels != -100).unsqueeze(-1)  # [B, S-1, 1]
        
        if valid_mask.sum() > 0:
            student_log_probs = F.log_softmax(shift_student / T, dim=-1)
            teacher_probs = F.softmax(shift_teacher / T, dim=-1)
            
            kl_div = F.kl_div(student_log_probs, teacher_probs, reduction="none")
            kd_loss = (kl_div * valid_mask).sum() / valid_mask.sum() * (T ** 2)
            
            total_loss = (alpha * student_ce_loss) + ((1.0 - alpha) * kd_loss)
        else:
            total_loss = student_ce_loss
            
        total_loss.backward()
        student_optimizer.step()
        
        total_train_loss += total_loss.item()
        
        if (step + 1) % CFG["log_interval"] == 0 or (step + 1) == len(train_loader):
            print(f"Student KD | Epoch {epoch+1}/{student_epochs} | Step {step+1}/{len(train_loader)} | Total Loss: {total_loss.item():.4f} (CE: {student_ce_loss.item():.4f})")
            
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation
    student_model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for batch in dev_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = student_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_val_loss += outputs["loss"].item()

    avg_val_loss = total_val_loss / len(dev_loader)
    print(f"Student KD | Epoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_student_val_loss or epoch == 0:
        best_student_val_loss = avg_val_loss
        torch.save(student_model.state_dict(), STUDENT_CHECKPOINT_PATH)
        print(f"  [+] Saved Best Distilled Student Checkpoint (Val Loss: {best_student_val_loss:.4f})")
        
    print("-" * 75)

Loaded Best Teacher Checkpoint for Knowledge Distillation.

=== TASK 2: Knowledge Distillation to Student Model (3 Epochs) ===
Student KD | Epoch 1/3 | Step 20/3164 | Total Loss: 3.7915 (CE: 5.6101)
Student KD | Epoch 1/3 | Step 40/3164 | Total Loss: 3.2974 (CE: 5.0929)
Student KD | Epoch 1/3 | Step 60/3164 | Total Loss: 3.5423 (CE: 5.7221)
Student KD | Epoch 1/3 | Step 80/3164 | Total Loss: 3.3657 (CE: 5.2521)
Student KD | Epoch 1/3 | Step 100/3164 | Total Loss: 3.4183 (CE: 5.4834)
Student KD | Epoch 1/3 | Step 120/3164 | Total Loss: 3.5530 (CE: 5.6598)
Student KD | Epoch 1/3 | Step 140/3164 | Total Loss: 2.9997 (CE: 4.7656)
Student KD | Epoch 1/3 | Step 160/3164 | Total Loss: 3.1523 (CE: 5.1055)
Student KD | Epoch 1/3 | Step 180/3164 | Total Loss: 2.8918 (CE: 4.6358)
Student KD | Epoch 1/3 | Step 200/3164 | Total Loss: 3.3723 (CE: 5.3966)
Student KD | Epoch 1/3 | Step 220/3164 | Total Loss: 2.9187 (CE: 4.6223)
Student KD | Epoch 1/3 | Step 240/3164 | Total Loss: 3.0117 (CE: 4.8567)
S

In [8]:
@torch.no_grad()
def generate_summary(model, tokenizer, prompt_text, max_new_tokens=256, temperature=0.7, top_p=0.9, device="cuda"):
    """
    Autoregressive text generation for legal document summarization.
    """
    model.eval()
    input_ids = tokenizer.encode(prompt_text, truncation=True, max_length=CFG["max_doc_len"], return_tensors="pt").to(device)
    
    generated = input_ids.clone()
    
    for _ in range(max_new_tokens):
        outputs = model(input_ids=generated, attention_mask=torch.ones_like(generated))
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs
        next_token_logits = logits[:, -1, :] / max(temperature, 1e-5)
        
        sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0
        
        indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
        next_token_logits[indices_to_remove] = float("-inf")
        
        probs = F.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        generated = torch.cat([generated, next_token], dim=1)
        
        if next_token.item() == tokenizer.eos_token_id:
            break
            
    prompt_len = input_ids.shape[1]
    gen_tokens = generated[0, prompt_len:].cpu().tolist()
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

print("Autoregressive generation helper configured successfully.")

Autoregressive generation helper configured successfully.


In [9]:
# ==============================================================================
# CELL 8.5: TEACHER MODEL INFERENCE & SUBMISSION GENERATION
# ==============================================================================
print("Reloading Best Teacher Checkpoint for Phase 6 Submission Generation...")

# 1. Reload Best Teacher Checkpoint Weights
if os.path.exists(TEACHER_CHECKPOINT_PATH):
    checkpoint = torch.load(TEACHER_CHECKPOINT_PATH, map_location=device)
    teacher_model.load_state_dict(checkpoint["trainable_state_dict"], strict=False)
    print(f"Successfully reloaded Teacher Model checkpoint.")
else:
    print("Warning: Teacher checkpoint path not found! Using current model weights.")

# 2. Prepare Test Set Dataset
test_dataset = LegalSummarizationDataset(test_data, tokenizer, max_doc_len=CFG["max_doc_len"], is_test=True)

teacher_model.eval()
predictions_teacher = []

print(f"Generating Abstractive Summaries using Teacher Model for {len(test_dataset)} Test Documents...")

for idx, sample in enumerate(test_dataset):
    doc_id = sample["id"]
    prompt_text = sample["prompt_text"]
    
    generated_text = generate_summary(
        model=teacher_model,
        tokenizer=tokenizer,
        prompt_text=prompt_text,
        max_new_tokens=CFG["gen_max_new_tokens"],
        device=device
    )
    
    # Fallback to prevent empty predictions
    if not generated_text:
        generated_text = "The legal document provides rulings and judicial findings on the case."
        
    predictions_teacher.append({
        "id": doc_id,
        "summary": generated_text
    })
    
    if (idx + 1) % CFG["log_interval"] == 0 or (idx + 1) == len(test_dataset):
        print(f"Teacher Generated {idx+1}/{len(test_dataset)} summaries.")

# 3. Output Serialization & Submission Packaging for Teacher Model
output_dir_teacher = "./phase6_teacher_lorma_summ"
os.makedirs(output_dir_teacher, exist_ok=True)
pred_path_teacher = os.path.join(output_dir_teacher, "prediction.json")
sub_zip_path_teacher = os.path.join(output_dir_teacher, "submission.zip")

with open(pred_path_teacher, "w", encoding="utf-8") as f:
    json.dump(predictions_teacher, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(sub_zip_path_teacher, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(pred_path_teacher, arcname="prediction.json")

print(f"\nPhase 6 Teacher Summarization Prediction Complete! Generated {len(predictions_teacher)} summaries.")
print(f"Teacher Submission saved to: {sub_zip_path_teacher}")

if DEBUG:
    print("\n[DEBUG] Sample Teacher Abstractive Summary Output:")
    print(json.dumps(predictions_teacher[:2], indent=2))

Reloading Best Teacher Checkpoint for Phase 6 Submission Generation...
Successfully reloaded Teacher Model checkpoint.
Generating Abstractive Summaries using Teacher Model for 100 Test Documents...
Teacher Generated 20/100 summaries.
Teacher Generated 40/100 summaries.
Teacher Generated 60/100 summaries.
Teacher Generated 80/100 summaries.
Teacher Generated 100/100 summaries.

Phase 6 Teacher Summarization Prediction Complete! Generated 100 summaries.
Teacher Submission saved to: ./phase6_teacher_lorma_summ/submission.zip


In [10]:
print("Reloading Best Distilled Student Checkpoint for Phase 6 Submission Generation...")

if os.path.exists(STUDENT_CHECKPOINT_PATH):
    student_model.load_state_dict(torch.load(STUDENT_CHECKPOINT_PATH, map_location=device))
    print(f"Successfully reloaded Distilled Student Model checkpoint.\n")

test_dataset = LegalSummarizationDataset(test_data, tokenizer, max_doc_len=CFG["max_doc_len"], is_test=True)

student_model.eval()
predictions = []

print(f"Generating Summaries using Distilled Student Model for {len(test_dataset)} Test Documents...")

for idx, sample in enumerate(test_dataset):
    doc_id = sample["id"]
    prompt_text = sample["prompt_text"]
    
    generated_text = generate_summary(
        model=student_model,
        tokenizer=tokenizer,
        prompt_text=prompt_text,
        max_new_tokens=CFG["gen_max_new_tokens"],
        device=device
    )
    
    if not generated_text:
        generated_text = "The legal document provides rulings and judicial findings on the case."
        
    predictions.append({
        "id": doc_id,
        "summary": generated_text
    })
    
    if (idx + 1) % CFG["log_interval"] == 0 or (idx + 1) == len(test_dataset):
        print(f"Generated {idx+1}/{len(test_dataset)} student summaries.")

# Output serialization and zip packaging according to Phase 6 requirements[cite: 12, 13]
output_dir = "./phase6_student_kd_summ"
os.makedirs(output_dir, exist_ok=True)
pred_path = os.path.join(output_dir, "prediction.json")
sub_zip_path = os.path.join(output_dir, "submission.zip")

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(sub_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(pred_path, arcname="prediction.json")

print(f"\nPhase 6 Distilled Student Summarization Prediction Complete! Generated {len(predictions)} summaries[cite: 10, 11].")
print(f"Submission saved to: {sub_zip_path}")

if DEBUG:
    print("\n[DEBUG] Sample Abstractive Summary Output:")
    print(json.dumps(predictions[:2], indent=2))

Reloading Best Distilled Student Checkpoint for Phase 6 Submission Generation...
Successfully reloaded Distilled Student Model checkpoint.

Generating Summaries using Distilled Student Model for 100 Test Documents...
Generated 20/100 student summaries.
Generated 40/100 student summaries.
Generated 60/100 student summaries.
Generated 80/100 student summaries.
Generated 100/100 student summaries.

Phase 6 Distilled Student Summarization Prediction Complete! Generated 100 summaries[cite: 10, 11].
Submission saved to: ./phase6_student_kd_summ/submission.zip
